<a href="https://colab.research.google.com/github/xD0ri4nx/bot_investitiV2/blob/main/bot_invetitii_collab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit yfinance scikit-learn tensorflow pandas numpy google-genai datasets transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 90.3 MB/s eta 0:00:00


In [2]:
import os
from google.colab import drive

# 1. Mount with a specific mount point
drive.mount('/content/drive')

# 2. Define your project path
PROJECT_PATH = '/content/drive/MyDrive/sentiment_csv'

# 3. Verification check
if os.path.exists(PROJECT_PATH):
    print(f"Success! Connected to: {PROJECT_PATH}")
    print(f"Files found: {os.listdir(PROJECT_PATH)}")

Mounted at /content/drive
Success! Connected to: /content/drive/MyDrive/sentiment_csv
Files found: ['NVDA_sentiment.csv', 'MSFT_sentiment.csv', 'GOOGL_sentiment.csv', 'SPY_sentiment.csv', 'BTC-USD_sentiment.csv', 'AAPL_sentiment.csv', 'ETH-USD_sentiment.csv']


In [ ]:
import pandas as pd
import torch
import gc
from datasets import load_dataset
from transformers import pipeline
from google.colab import userdata

# ==============================================================================
# 1. SECURE CONFIGURATION & LOCAL NLP INITIALIZATION
# ==============================================================================
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("[CRITICAL ERROR] Secret 'HF_TOKEN' not found. Please add it to Colab Secrets.")
    raise

# NOTE: Adjust this list! If you only need Apple right now, change to: TICKERS = ["AAPL"]
TICKERS = ["ETH-USD"]

# Institutional Backtest Target: ~10 years of data
MAX_ARTICLES_PER_ASSET = 10000

device = 0 if torch.cuda.is_available() else -1
if device == 0:
    print("🚀 GPU Detected! FinBERT will run with hardware acceleration.")
else:
    print("⚠️ No GPU detected. FinBERT will run on CPU (slower).")

print("Loading FinBERT Financial NLP Model into local memory...")
sentiment_analyzer = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=device)
print("FinBERT Model Loaded Successfully!")

# ==============================================================================
# 2. FINBERT REASONING ENGINE (UPGRADED WITH GPU BATCHING)
# ==============================================================================
def analyze_daily_news(headlines_list):
    """Scores headlines using GPU Batching (batch_size=16) for maximum speed."""
    if not headlines_list:
        return 0.0

    # Truncate to 512 chars to prevent model indexing errors on massive articles
    safe_headlines = [str(h)[:512] for h in headlines_list]

    daily_score = 0.0
    valid_articles = 0

    try:
        # THE BATCHING OPTIMIZATION: Process 16 articles simultaneously on the GPU
        results = sentiment_analyzer(safe_headlines, batch_size=16)

        for result in results:
            label = result['label']
            if label == 'positive':
                daily_score += 1.0
            elif label == 'negative':
                daily_score += -1.0
            else:
                daily_score += 0.0

            valid_articles += 1

    except Exception as e:
        # If a severely corrupted batch crashes the pipeline, skip it safely
        return 0.0

    if valid_articles > 0:
        return round(daily_score / valid_articles, 4)
    return 0.0

# ==============================================================================
# 3. FAULT-TOLERANT DATA STREAMING PIPELINE (WITH CIRCUIT BREAKER)
# ==============================================================================
def fetch_raw_historical_news(ticker):
    print(f"\nStreaming Massive Hugging Face dataset for {ticker}...")

    ASSET_NAMES = {
        "AAPL": "apple",
        "NVDA": "nvidia",
        "MSFT": "microsoft",
        "GOOGL": "google",
        "SPY": "s&p 500",
        "BTC-USD": "bitcoin",
        "ETH-USD": "ethereum"
    }

    search_name = ASSET_NAMES.get(ticker, ticker.lower())

    try:
        dataset = load_dataset(
            "Brianferrell787/financial-news-multisource",
            split="train",
            streaming=True,
            token=HF_TOKEN
        )
    except Exception as e:
        print(f"[!] HuggingFace connection error: {e}")
        return {}

    news_dict = {}
    extracted_count = 0
    rows_scanned = 0

    # NEW: THE CIRCUIT BREAKER
    # If the script reads 3 million rows, it forces a stop to prevent RAM crashes.
    MAX_SCAN_DEPTH = 3000000

    try:
        for row in dataset:
            rows_scanned += 1
            text = str(row.get('text', ''))
            extra = str(row.get('extra_fields', ''))

            if search_name in text.lower() or ticker in extra:
                raw_date = row.get('date')
                try:
                    clean_date = pd.to_datetime(raw_date).strftime('%Y-%m-%d')
                except:
                    continue

                if clean_date not in news_dict:
                    news_dict[clean_date] = []

                news_dict[clean_date].append(text)
                extracted_count += 1

                if extracted_count % 500 == 0:
                    print(f"   ...Found {extracted_count}/{MAX_ARTICLES_PER_ASSET} articles for {ticker}...")

                if extracted_count >= MAX_ARTICLES_PER_ASSET:
                    break

            # The Circuit Breaker Trigger
            if rows_scanned >= MAX_SCAN_DEPTH:
                print(f"\n   [!] WARNING: Scanned {MAX_SCAN_DEPTH} rows. Stopping search to protect Colab RAM.")
                break

    except Exception as e:
        print(f"\n   [!] HUGGING FACE SERVER CRASH INTERCEPTED: {e}")

    print(f"   -> Saving the {extracted_count} articles we successfully grabbed.")
    print(f"[SUCCESS] Extracted {extracted_count} articles across {len(news_dict)} trading days for {ticker}.")
    return news_dict

# ==============================================================================
# 4. MEMORY-SAFE BATCH EXECUTION
# ==============================================================================
def run_sentiment_miner():
    print("\n--- STARTING OPTIMIZED NLP PIPELINE ---")

    for ticker in TICKERS:
        print(f"\n================ Processing {ticker} ================")
        historical_news = fetch_raw_historical_news(ticker)

        results = []
        total_days = len(historical_news)

        if total_days == 0:
            print(f"⚠️ [WARNING] No data found for {ticker}, skipping.")
            continue

        print(f"[{ticker}] Scoring {total_days} days using GPU Batching... Please wait.")

        for date, headlines in historical_news.items():
            score = analyze_daily_news(headlines)
            results.append({"Date": date, "Sentiment_Score": score})

        df_results = pd.DataFrame(results)
        df_results['Date'] = pd.to_datetime(df_results['Date'])
        df_results.sort_values('Date', inplace=True)

        output_filename = f"{ticker}_sentiment.csv"
        df_results.to_csv(output_filename, index=False)
        print(f"✅ [SAVED] {output_filename} is ready! (Total days scored: {total_days})")

        # ----------------------------------------------------------------------
        # GARBAGE COLLECTION TO PREVENT RAM CRASHES
        # ----------------------------------------------------------------------
        print(f"🧹 Sweeping memory logs to protect Colab RAM...")
        del historical_news
        del results
        del df_results
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        # ----------------------------------------------------------------------

run_sentiment_miner()

🚀 GPU Detected! FinBERT will run with hardware acceleration.
Loading FinBERT Financial NLP Model into local memory...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

Resolving data files:   0%|          | 0/131 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/131 [00:00<?, ?it/s]

   ...Found 500/10000 articles for ETH-USD...
   ...Found 1000/10000 articles for ETH-USD...
   ...Found 1500/10000 articles for ETH-USD...
   ...Found 2000/10000 articles for ETH-USD...

   [!] WARNING: Scanned 3000000 rows. Stopping search to protect Colab RAM.
   -> Saving the 2139 articles we successfully grabbed.
[SUCCESS] Extracted 2139 articles across 813 trading days for ETH-USD.
[ETH-USD] Scoring 813 days using GPU Batching... Please wait.


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ [SAVED] ETH-USD_sentiment.csv is ready! (Total days scored: 813)
🧹 Sweeping memory logs to protect Colab RAM...


In [5]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import RobustScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, Input, Bidirectional, Conv1D, GRU, LSTM,
    BatchNormalization, GaussianNoise, LayerNormalization, Add,
    MultiHeadAttention, GlobalAveragePooling1D, Multiply,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision
from tensorflow.keras.regularizers import l2
import warnings
import os
import gc

# ─────────────────────────────────────────────────────────────
# 1. SYSTEM CONFIG
# ─────────────────────────────────────────────────────────────
tf.random.set_seed(42)
np.random.seed(42)
mixed_precision.set_global_policy(mixed_precision.Policy('mixed_float16'))

for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

st.set_page_config(
    page_title="Institutional Quant AI v2",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
    .stApp{background:linear-gradient(135deg,#0b132b,#1c2541,#3a506b);color:#fff}
    header[data-testid="stHeader"]{background:transparent!important}
    [data-testid="stSidebar"]{background:rgba(28,37,65,.4)!important;backdrop-filter:blur(15px);border-right:1px solid rgba(255,255,255,.1)}
    [data-testid="stMetric"]{background:rgba(255,255,255,.05);backdrop-filter:blur(12px);border:1px solid rgba(255,255,255,.1);border-radius:12px;padding:20px;box-shadow:0 8px 32px rgba(0,0,0,.3);transition:transform .2s,box-shadow .2s}
    [data-testid="stMetric"]:hover{transform:translateY(-2px);box-shadow:0 12px 40px rgba(0,0,0,.4);background:rgba(255,255,255,.08)}
    [data-testid="stMetricValue"]{color:#fff!important}
    div.stButton>button{background:linear-gradient(90deg,#3a506b,#1c2541);color:#fff;border:1px solid rgba(255,255,255,.2);border-radius:8px;padding:10px 24px;font-weight:600;letter-spacing:.5px;transition:all .3s;box-shadow:0 4px 15px rgba(0,0,0,.2);width:100%}
    div.stButton>button:hover{background:linear-gradient(90deg,#5bc0be,#3a506b);border:1px solid rgba(255,255,255,.5);box-shadow:0 0 20px rgba(91,192,190,.4);transform:translateY(-1px)}
    h1,h2,h3,p,span,div,label{color:#e0e0e0}
    *:focus,*:focus-visible,*:focus-within{outline:none!important}
    div[data-baseweb="select"]>div,div[data-baseweb="input"]>div{background:linear-gradient(90deg,#1c2541,#3a506b)!important;border:1px solid rgba(255,255,255,.2)!important;border-radius:8px!important}
    div[data-baseweb="select"]>div:focus-within,div[data-baseweb="input"]>div:focus-within{border-color:#5bc0be!important}
    div[data-baseweb="select"] *,div[data-baseweb="input"] input{color:#fff!important}
    div[role="radiogroup"]>label{color:#e0e0e0!important}
    .cfg-badge{background:rgba(91,192,190,.15);border:1px solid rgba(91,192,190,.4);border-radius:6px;padding:8px 12px;font-size:12px;color:#5bc0be;margin-bottom:8px;display:block}
</style>
""", unsafe_allow_html=True)

st.title("⚡ Institutional Quant AI v2 — Adaptive Configuration Engine")

# ─────────────────────────────────────────────────────────────
# 2. COMPREHENSIVE PER-ASSET OPTIMAL CONFIGURATIONS
# ─────────────────────────────────────────────────────────────
ARCHS = [
    "TCN + MHA + GRU (Sniper Alpha v2)",
    "BiLSTM + Attention (Deep Memory v2)",
    "TFT-Lite (Quantum Fusion)",
]

IDEAL_CONFIGS = {
    "AAPL": {"arch": "TCN + MHA + GRU (Sniper Alpha v2)", "start": "2015-01-01", "time_step": 60, "ensemble_size": 6, "mc_passes": 20, "long_only": True, "cost_bps": 8, "target_vol": 28, "threshold": 0.18, "use_kelly": True, "reason": "Strong trend-momentum; TCN+MHA captures multi-scale acceleration"},
    "NVDA": {"arch": "TFT-Lite (Quantum Fusion)", "start": "2018-01-01", "time_step": 90, "ensemble_size": 8, "mc_passes": 30, "long_only": True, "cost_bps": 10, "target_vol": 60, "threshold": 0.28, "use_kelly": True, "reason": "Explosive regime shifts; TFT variable-selection adapts per phase"},
    "MSFT": {"arch": "BiLSTM + Attention (Deep Memory v2)", "start": "2015-01-01", "time_step": 50, "ensemble_size": 5, "mc_passes": 20, "long_only": True, "cost_bps": 8, "target_vol": 22, "threshold": 0.15, "use_kelly": True, "reason": "Smooth compounder; BiLSTM memory handles slow fundamental cycles"},
    "GOOGL": {"arch": "TCN + MHA + GRU (Sniper Alpha v2)", "start": "2015-01-01", "time_step": 60, "ensemble_size": 6, "mc_passes": 20, "long_only": True, "cost_bps": 8, "target_vol": 28, "threshold": 0.18, "use_kelly": True, "reason": "Episodic momentum around AI/ad cycles; TCN dilation optimal"},
    "SPY": {"arch": "BiLSTM + Attention (Deep Memory v2)", "start": "2010-01-01", "time_step": 40, "ensemble_size": 8, "mc_passes": 25, "long_only": True, "cost_bps": 3, "target_vol": 14, "threshold": 0.08, "use_kelly": True, "reason": "Most efficient instrument; BiLSTM captures macro mean-reversion"},
    "BTC-USD": {"arch": "TFT-Lite (Quantum Fusion)", "start": "2019-01-01", "time_step": 90, "ensemble_size": 8, "mc_passes": 35, "long_only": False, "cost_bps": 18, "target_vol": 75, "threshold": 0.38, "use_kelly": True, "reason": "Halving-cycle regimes; TFT variable-selection adapts bull/bear"},
    "ETH-USD": {"arch": "TFT-Lite (Quantum Fusion)", "start": "2019-01-01", "time_step": 90, "ensemble_size": 9, "mc_passes": 40, "long_only": False, "cost_bps": 20, "target_vol": 90, "threshold": 0.42, "use_kelly": True, "reason": "DeFi narrative shifts; TFT + max ensemble handles extreme vol"},
}

# ─────────────────────────────────────────────────────────────
# 3. SESSION STATE
# ─────────────────────────────────────────────────────────────
if "last_ticker" not in st.session_state:
    st.session_state.last_ticker = None

def apply_config(ticker: str):
    c = IDEAL_CONFIGS[ticker]
    st.session_state["cfg_arch"]      = c["arch"]
    st.session_state["cfg_start"]     = pd.to_datetime(c["start"]).date()
    st.session_state["cfg_time_step"] = c["time_step"]
    st.session_state["cfg_ensemble"]  = c["ensemble_size"]
    st.session_state["cfg_mc"]        = c["mc_passes"]
    st.session_state["cfg_longonly"]  = c["long_only"]
    st.session_state["cfg_cost"]      = c["cost_bps"]
    st.session_state["cfg_vol"]       = c["target_vol"]
    st.session_state["cfg_thresh"]    = float(c["threshold"])
    st.session_state["cfg_kelly"]     = c["use_kelly"]

# ─────────────────────────────────────────────────────────────
# 4. SIDEBAR
# ─────────────────────────────────────────────────────────────
with st.sidebar:
    st.header("1. Asset Class")
    ticker = st.selectbox("Select Asset", list(IDEAL_CONFIGS.keys()), key="ticker_select")

    if ticker != st.session_state.last_ticker:
        apply_config(ticker)
        st.session_state.last_ticker = ticker

    cfg = IDEAL_CONFIGS[ticker]
    st.markdown(f'<span class="cfg-badge">🧠 {cfg["reason"]}</span>', unsafe_allow_html=True)

    st.markdown("---")
    st.header("2. Neural Engine")
    model_arch = st.radio("Architecture:", ARCHS, index=ARCHS.index(st.session_state["cfg_arch"]), key="cfg_arch")

    st.markdown("---")
    st.header("3. Training Hyperparameters")
    time_step     = st.slider("Lookback window (days)", 30, 120, key="cfg_time_step", step=5)
    ensemble_size = st.slider("Ensemble members", 1, 10, key="cfg_ensemble")
    mc_passes     = st.slider("MC Dropout passes", 10, 50, key="cfg_mc")

    st.markdown("---")
    st.header("4. Date Range")
    start = st.date_input("Start Date", key="cfg_start")
    end   = st.date_input("End Date", value=pd.to_datetime("today").date())

    st.markdown("---")
    st.header("5. Risk Management")
    long_only         = st.checkbox("Long Only Mode", key="cfg_longonly")
    cost_bps          = st.slider("Transaction Cost (bps)", 0, 50, key="cfg_cost")
    target_vol        = st.slider("Target Volatility (%)", 10, 100, key="cfg_vol")
    conviction_thresh = st.slider("Conviction Deadband", 0.0, 1.0, key="cfg_thresh", step=0.01)
    use_kelly         = st.checkbox("Kelly Criterion sizing", key="cfg_kelly")

# ─────────────────────────────────────────────────────────────
# 5. QUANT PIPELINE
# ─────────────────────────────────────────────────────────────
class QuantPipeline:
    def __init__(self, symbol, start_date, end_date, time_step, model_architecture):
        self.symbol             = symbol
        self.start_date         = start_date
        self.end_date           = end_date
        self.time_step          = time_step
        self.model_architecture = model_architecture
        self.scaler             = RobustScaler()
        self.n_features         = None
        self.project_path       = '/content/drive/MyDrive/sentiment_csv'

    def fetch_data(self):
        try:
            df = yf.download(self.symbol, start=self.start_date, end=self.end_date, progress=False)
            if len(df) == 0:
                raise ValueError("No price data returned from Yahoo Finance.")

            if 'Date' not in df.columns and df.index.name == 'Date':
                df = df.reset_index()
            elif 'Date' not in df.columns and type(df.index) == pd.DatetimeIndex:
                df = df.reset_index()
                df.rename(columns={'index': 'Date'}, inplace=True)
            else:
                rename_map = {'date': 'Date', 'time': 'Date', 'begins_at': 'Date', 'Timestamp': 'Date'}
                df.rename(columns=rename_map, inplace=True)

            if 'Date' in df.columns:
                df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            if 'Adj Close' in df.columns and 'Close' not in df.columns:
                df['Close'] = df['Adj Close']

            sentiment_file = os.path.join(self.project_path, f"{self.symbol}_sentiment.csv")
            if not os.path.exists(sentiment_file):
                st.error(f"❌ Could not find {sentiment_file} in Drive. Make sure your Drive is mounted and the file exists!")
                st.stop()

            sentiment_df = pd.read_csv(sentiment_file)
            sentiment_df['Date'] = pd.to_datetime(sentiment_df['Date']).dt.tz_localize(None)

            merged_df = pd.merge(df, sentiment_df, on='Date', how='left')
            merged_df.sort_values('Date', inplace=True)

            if 'Sentiment_Score' in merged_df.columns:
                merged_df['Sentiment_Score'] = merged_df['Sentiment_Score'].fillna(0.0)

            merged_df.set_index('Date', inplace=True)

            return merged_df
        except Exception as e:
            st.error(f"Data Feed Error: {e}")
            st.stop()

    def engineer_features(self, df):
        c, h, l, v, o = (df['Close'], df['High'], df['Low'], df['Volume'], df['Open'])

        df['Log_Returns'] = np.log(c / c.shift(1))
        df['Target']      = df['Log_Returns'].shift(-1)

        df['Garman_Klass'] = (0.5 * np.log(h/l)**2 - (2*np.log(2)-1) * np.log(c/o)**2).clip(lower=0)

        delta = c.diff()
        gain  = delta.where(delta > 0, 0).rolling(14).mean()
        loss  = (-delta.where(delta < 0, 0)).rolling(14).mean()
        df['RSI'] = 100 - 100 / (1 + gain / loss.replace(0, 1e-9))
        rr = df['RSI'].rolling(14).max() - df['RSI'].rolling(14).min()
        df['Stoch_RSI'] = ((df['RSI'] - df['RSI'].rolling(14).min()) / rr.replace(0, 1e-9)).clip(0, 1)

        hl  = (h - l).replace(0, 1e-4)
        ad  = ((c - l) - (h - c)) / hl * v
        df['CMF'] = (ad.rolling(20).sum() / v.rolling(20).sum().replace(0, 1e-9)).fillna(0).clip(-1, 1)

        obv       = (np.sign(df['Log_Returns']) * v).cumsum()
        obv_ema   = obv.ewm(span=20, adjust=False).mean()
        df['OBV_Ratio'] = (obv / obv_ema.replace(0, 1e-9)).clip(-3, 3)

        ema12 = c.ewm(span=12, adjust=False).mean()
        ema26 = c.ewm(span=26, adjust=False).mean()
        macd  = ema12 - ema26
        df['MACD_Hist'] = macd - macd.ewm(span=9, adjust=False).mean()

        sma20 = c.rolling(20).mean()
        std20 = c.rolling(20).std().replace(0, 1e-9)
        df['BB_Position'] = ((c - (sma20 - 2*std20)) / (4*std20)).clip(0, 1)

        tr = pd.concat([h-l, (h-c.shift(1)).abs(), (l-c.shift(1)).abs()], axis=1).max(axis=1)
        df['ATR'] = (tr.rolling(14).mean() / c.replace(0, 1e-9)).clip(0, 0.5)

        hh = h.rolling(14).max()
        ll = l.rolling(14).min()
        df['Williams_R'] = ((hh - c) / (hh - ll).replace(0, 1e-9)).clip(0, 1)

        up_move = h - h.shift(1)
        dn_move = l.shift(1) - l
        pdm = up_move.where((up_move > dn_move) & (up_move > 0), 0)
        ndm = dn_move.where((dn_move > up_move) & (dn_move > 0), 0)
        atr14 = tr.rolling(14).mean().replace(0, 1e-9)
        pdi   = 100 * pdm.rolling(14).mean() / atr14
        ndi   = 100 * ndm.rolling(14).mean() / atr14
        dx    = 100 * (pdi - ndi).abs() / (pdi + ndi).replace(0, 1e-9)
        df['ADX'] = (dx.rolling(14).mean() / 100).clip(0, 1)

        typical = (h + l + c) / 3
        vwap = ((typical * v).rolling(20).sum() / v.rolling(20).sum().replace(0, 1e-9))
        df['VWAP_Dev'] = ((c - vwap) / vwap.replace(0, 1e-9)).clip(-0.3, 0.3)

        df['MOM_5']  = c.pct_change(5).clip(-0.5, 0.5)
        df['MOM_20'] = c.pct_change(20).clip(-0.5, 0.5)
        df['MOM_60'] = c.pct_change(60).clip(-1.0, 1.0)

        rv     = df['Log_Returns'].rolling(20).std() * np.sqrt(252)
        rv_mu  = rv.rolling(252).mean()
        rv_std = rv.rolling(252).std().replace(0, 1e-9)
        df['Vol_Regime'] = ((rv - rv_mu) / rv_std).clip(-3, 3)

        t = df.index
        df['Day_Sin']   = np.sin(2 * np.pi * t.dayofweek / 7)
        df['Day_Cos']   = np.cos(2 * np.pi * t.dayofweek / 7)
        df['Month_Sin'] = np.sin(2 * np.pi * t.month / 12)
        df['Month_Cos'] = np.cos(2 * np.pi * t.month / 12)

        df.dropna(inplace=True)
        return df

    def prepare_tensors(self, df, split_ratio=0.8):
        feature_cols = [
            'Sentiment_Score',
            'Log_Returns','Garman_Klass','RSI','Stoch_RSI','CMF','OBV_Ratio',
            'MACD_Hist','BB_Position','ATR','Williams_R','ADX','VWAP_Dev',
            'MOM_5','MOM_20','MOM_60','Vol_Regime',
            'Day_Sin','Day_Cos','Month_Sin','Month_Cos'
        ]
        self.n_features = len(feature_cols)

        split_idx = int(len(df) * split_ratio)
        train_df  = df.iloc[:split_idx]
        test_df   = df.iloc[split_idx:]

        decay   = np.exp(0.0015 * np.arange(len(train_df)))
        weights = decay / decay.mean()

        X_tr_sc = self.scaler.fit_transform(train_df[feature_cols]).astype(np.float32)
        X_te_sc = self.scaler.transform(test_df[feature_cols]).astype(np.float32)

        y_tr = train_df['Target'].values.astype(np.float32)
        y_te = test_df['Target'].values.astype(np.float32)

        def window(X, y, ts, w=None):
            Xs, ys, ws = [], [], []
            for i in range(len(X) - ts):
                Xs.append(X[i:i+ts])
                ys.append(y[i+ts])
                if w is not None:
                    ws.append(w[i+ts])
            Xs = np.array(Xs)
            ys = np.array(ys)
            return (Xs, ys, np.array(ws)) if w is not None else (Xs, ys)

        X_tr, y_tr, w_tr = window(X_tr_sc, y_tr, self.time_step, weights)
        X_te, y_te       = window(X_te_sc, y_te, self.time_step)
        return X_tr, y_tr, w_tr, X_te, y_te, test_df.iloc[self.time_step:]

    def _assemble_model(self, input_shape):
        tf.keras.backend.clear_session()
        gc.collect()

        inputs = Input(shape=input_shape, name="seq_input")
        x = GaussianNoise(0.005)(inputs)
        x = BatchNormalization()(x)

        if self.model_architecture == "TCN + MHA + GRU (Sniper Alpha v2)":
            c1 = Conv1D(64, 3, dilation_rate=1, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(x)
            c2 = Conv1D(64, 3, dilation_rate=2, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(c1)
            c4 = Conv1D(64, 3, dilation_rate=4, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(c2)
            c8 = Conv1D(64, 3, dilation_rate=8, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(c4)
            tcn  = LayerNormalization()(Add()([c1, c8]))
            attn = MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.1)(tcn, tcn)
            attn = LayerNormalization()(Add()([tcn, attn]))
            x = Bidirectional(GRU(128, return_sequences=False, kernel_regularizer=l2(1e-4)))(attn)
            x = Dropout(0.4)(x)

        elif self.model_architecture == "BiLSTM + Attention (Deep Memory v2)":
            x = Bidirectional(LSTM(128, return_sequences=True, kernel_regularizer=l2(1e-4)))(x)
            x = LayerNormalization()(x)
            x = Dropout(0.35)(x)
            x = Bidirectional(LSTM(64, return_sequences=True, kernel_regularizer=l2(1e-4)))(x)
            x = LayerNormalization()(x)
            x = Dropout(0.35)(x)
            score   = Dense(1)(x)
            weights = tf.nn.softmax(score, axis=1)
            x = tf.reduce_sum(x * weights, axis=1)
            x = Dropout(0.3)(x)

        else:   # TFT-Lite
            def grn(z, units=64, drop=0.1, pfx="g"):
                skip = z
                z    = Dense(units, activation='elu', kernel_regularizer=l2(1e-4), name=f"{pfx}_d1")(z)
                z    = Dense(units, kernel_regularizer=l2(1e-4), name=f"{pfx}_d2")(z)
                gate = Dense(units, activation='sigmoid', name=f"{pfx}_gate")(z)
                z    = Multiply(name=f"{pfx}_mul")([z, gate])
                z    = Dropout(drop, name=f"{pfx}_drop")(z)
                if skip.shape[-1] != units:
                    skip = Dense(units, name=f"{pfx}_skip")(skip)
                return LayerNormalization(name=f"{pfx}_ln")(Add(name=f"{pfx}_add")([z, skip]))

            vsn_in = Dense(64, activation='elu', name="vsn_proj")(x)
            vsn_w  = Dense(self.n_features, activation='softmax', name="vsn_w")(
                GlobalAveragePooling1D(name="vsn_pool")(vsn_in)[:, tf.newaxis, :])
            x_sel  = Dense(64, name="vsn_combine")(
                tf.reduce_sum(tf.expand_dims(x, -1) * tf.reshape(vsn_w, (-1, 1, self.n_features, 1)), axis=2))
            x_grn  = grn(vsn_in + x_sel, units=64, pfx="grn1")
            attn   = MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.1, name="t_mha")(x_grn, x_grn)
            attn   = LayerNormalization(name="attn_ln")(Add(name="attn_add")([x_grn, attn]))
            attn   = grn(attn, units=64, drop=0.2, pfx="grn2")
            x      = GlobalAveragePooling1D(name="gap")(attn)
            x      = Dropout(0.3, name="gap_drop")(x)

        x   = Dense(128, activation='gelu', kernel_regularizer=l2(1e-4), name="head1")(x)
        x   = Dropout(0.3, name="drop1")(x)
        x   = Dense(64,  activation='gelu', kernel_regularizer=l2(1e-4), name="head2")(x)
        x   = Dropout(0.2, name="drop2")(x)
        out = Dense(1, activation='linear', dtype='float32', name="output")(x)

        model = Model(inputs=inputs, outputs=out)
        model.compile(optimizer=Adam(learning_rate=0.0005, clipnorm=1.0), loss='huber')
        return model

    def mc_predict(self, model, X, n_passes=20, batch_size=256):
        preds = []
        ds = (tf.data.Dataset.from_tensor_slices(X).batch(batch_size).prefetch(tf.data.AUTOTUNE))
        for _ in range(n_passes):
            p = np.concatenate([model(b, training=True).numpy().flatten() for b in ds])
            preds.append(p)
        preds = np.array(preds)
        return preds.mean(axis=0), preds.std(axis=0)


# ─────────────────────────────────────────────────────────────
# 6. LAUNCH BUTTON
# ─────────────────────────────────────────────────────────────
arch_short = model_arch.split("(")[1].rstrip(")")

if st.button(f"🚀 Launch {arch_short} — {ticker}"):

    pipeline = QuantPipeline(ticker, start, end, time_step, model_arch)

    with st.status(f"⚡ Running {arch_short} on {ticker} …", expanded=True) as status:

        st.write("📡 Fetching market data & merging NLP Drive sentiment scores…")
        raw_df = pipeline.fetch_data()

        st.write("🔬 Engineering 20+ alpha factors …")
        proc_df = pipeline.engineer_features(raw_df)
        st.write(f"   {len(proc_df):,} trading days · {pipeline.n_features} features")

        st.write("⚙️  Building tensors with exponential decay weights …")
        X_tr, y_tr, w_tr, X_te, y_te, test_ctx = pipeline.prepare_tensors(proc_df)

        BATCH = 256
        train_ds = (tf.data.Dataset.from_tensor_slices((X_tr, y_tr, w_tr))
                    .shuffle(2048).batch(BATCH).prefetch(tf.data.AUTOTUNE))
        test_ds  = (tf.data.Dataset.from_tensor_slices((X_te, y_te))
                    .batch(BATCH).prefetch(tf.data.AUTOTUNE))

        st.write(f"🧠 Training {ensemble_size}-member ensemble …")
        ens_preds, ens_uncerts, val_losses = [], [], []
        bar = st.progress(0)

        for i in range(ensemble_size):
            tf.random.set_seed(42 + i * 7)
            model = pipeline._assemble_model((X_tr.shape[1], X_tr.shape[2]))
            hist  = model.fit(
                train_ds,
                validation_data=test_ds,
                epochs=80,
                callbacks=[
                    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, min_delta=1e-6),
                    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-7, verbose=0)
                ],
                verbose=0
            )
            best = min(hist.history['val_loss'])
            val_losses.append(best)
            mu, sigma = pipeline.mc_predict(model, X_te, mc_passes, BATCH)
            ens_preds.append(mu)
            ens_uncerts.append(sigma)
            st.write(f"   ✓ Member {i+1}/{ensemble_size}  val_loss={best:.6f}")
            bar.progress((i + 1) / ensemble_size)

        inv = 1.0 / (np.array(val_losses) + 1e-9)
        ew  = inv / inv.sum()
        avg_preds  = np.average(ens_preds,   axis=0, weights=ew)
        avg_uncert = np.average(ens_uncerts, axis=0, weights=ew)

        status.update(label="✅ Complete", state="complete", expanded=False)

    # ── Backtest ─────────────────────────────────────────────
    bt = test_ctx.copy()
    bt['Predicted_Return'] = avg_preds
    bt['Uncertainty']      = avg_uncert
    bt['Actual_Return']    = y_te

    # 1. MACRO TREND IDENTIFICATION
    bt['EMA_20'] = bt['Close'].ewm(span=20, adjust=False).mean()
    bt['SMA_50'] = bt['Close'].rolling(50).mean()
    bt['Macro_Bull'] = bt['EMA_20'] > bt['SMA_50']

    # 2. ASYMMETRIC VOLATILITY TARGET (Don't penalize upside breakouts)
    cur_vol = (np.sqrt(bt['Garman_Klass'].clip(lower=0)) * np.sqrt(252)).replace(0, 0.01)
    dynamic_target_vol = np.where(bt['Macro_Bull'], target_vol * 1.5, target_vol)
    bt['Vol_Scalar'] = (dynamic_target_vol / (cur_vol * 100)).clip(0.2, 3.0)

    # 3. AI PREDICTION SCALING (Trend Alignment)
    # Boost AI conviction heavily if it aligns with the macro trend
    trend_alignment = np.where(bt['Macro_Bull'] & (bt['Predicted_Return'] > 0), 2.5, 1.0)
    sig = tf.math.sigmoid(bt['Predicted_Return'] * 500 * trend_alignment).numpy() - 0.5
    bt['Conviction'] = (sig * 2).clip(-1, 1)

    # Apply minor penalty for high neural network uncertainty
    unc_norm = bt['Uncertainty'] / (bt['Uncertainty'].max() + 1e-9)
    bt['Conviction'] *= (1 - 0.5 * unc_norm)

    # 4. DYNAMIC DEADBAND (The "Stay in the Trade" Fix)
    # If we are in a massive run, lower the deadband significantly to stay in the trade!
    dynamic_thresh = np.where(bt['Macro_Bull'], conviction_thresh * 0.3, conviction_thresh)
    bt['Conviction']  = np.where(bt['Conviction'].abs() >= dynamic_thresh, bt['Conviction'], 0.0)

    if use_kelly:
        roll = 60
        correct = (np.sign(bt['Predicted_Return']) == np.sign(bt['Actual_Return'])).astype(float)
        kp  = correct.shift(1).rolling(roll).mean().fillna(0.5).clip(0.01, 0.99)
        pr  = bt['Actual_Return'].where(bt['Actual_Return'] > 0, np.nan).shift(1).rolling(roll).mean().fillna(0.001)
        nr  = bt['Actual_Return'].where(bt['Actual_Return'] < 0, np.nan).abs().shift(1).rolling(roll).mean().fillna(0.001)
        kb  = (pr / nr.replace(0, 1e-9)).clip(0.1, 5)
        kf  = (0.5 * (kp - (1 - kp) / kb)).clip(0, 1)
        raw = bt['Conviction'] * bt['Vol_Scalar'] * kf
    else:
        raw = bt['Conviction'] * bt['Vol_Scalar']

    bt['Position'] = (raw.clip(lower=0) if long_only else raw).fillna(0).clip(-3, 3)
    bt['Turnover'] = bt['Position'].diff().abs().fillna(0)
    bt['Cost']     = bt['Turnover'] * (cost_bps / 10000)
    bt['Gross_Return'] = bt['Position'] * bt['Actual_Return']
    bt['Net_Return']   = bt['Gross_Return'] - bt['Cost']
    bt['Cum_Market']   = (1 + bt['Actual_Return']).cumprod()
    bt['Cum_Strategy'] = (1 + bt['Net_Return']).cumprod()

    ann     = 252
    mu_r    = bt['Net_Return'].mean()
    sd_r    = bt['Net_Return'].std() + 1e-12
    sharpe  = float(np.sqrt(ann) * mu_r / sd_r)
    down    = bt['Net_Return'].where(bt['Net_Return'] < 0, 0).std() + 1e-12
    sortino = float(np.sqrt(ann) * mu_r / down)
    peak    = bt['Cum_Strategy'].cummax()
    max_dd  = float(((bt['Cum_Strategy'] - peak) / peak).min())
    ann_ret = float(bt['Cum_Strategy'].iloc[-1] ** (ann / max(len(bt), 1)) - 1)
    calmar  = ann_ret / (abs(max_dd) + 1e-12)
    total   = float(bt['Cum_Strategy'].iloc[-1] - 1)

    bt['Dir_Correct'] = (np.sign(bt['Predicted_Return']) == np.sign(bt['Actual_Return'])).astype(float)
    active   = bt[bt['Position'].abs() > 0]
    win_rate = len(active[active['Net_Return'] > 0]) / max(len(active), 1)
    accuracy = active['Dir_Correct'].mean() * 100 if len(active) > 0 else 0.0

    # ── Display ───────────────────────────────────────────────
    st.divider()
    st.subheader(f"📊 {ticker} — {arch_short}")
    st.markdown(f'<span class="cfg-badge">⚙️ Optimal config applied: {cfg["reason"]}</span>', unsafe_allow_html=True)

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Net Total Return", f"{total*100:.2f}%")
    c2.metric("Sharpe Ratio",     f"{sharpe:.2f}")
    c3.metric("Sortino Ratio",    f"{sortino:.2f}")
    c4.metric("Calmar Ratio",     f"{calmar:.2f}")
    st.markdown("<br>", unsafe_allow_html=True)
    c5, c6, c7, c8 = st.columns(4)
    c5.metric("Max Drawdown",          f"{max_dd*100:.2f}%")
    c6.metric("Win Rate",              f"{win_rate*100:.1f}%")
    c7.metric("Directional Accuracy",  f"{accuracy:.1f}%")
    c8.metric("Avg Uncertainty",       f"±{avg_uncert.mean()*100:.3f}%")

    st.markdown("<br>", unsafe_allow_html=True)
    st.subheader("Equity Curve (net of fees)")
    st.line_chart(bt[['Cum_Strategy','Cum_Market']].rename(columns={'Cum_Strategy':'AI Strategy','Cum_Market':'Buy & Hold'}))

    st.subheader("Model Uncertainty over Time")
    st.area_chart(pd.Series(avg_uncert * 100, index=bt.index[:len(avg_uncert)], name="Prediction Std (%)"))

    with st.expander("📐 Ensemble Member Details"):
        st.dataframe(pd.DataFrame({"Member": [f"Model {i+1}" for i in range(ensemble_size)], "Val Loss": [f"{v:.6f}" for v in val_losses], "Weight": [f"{w:.4f}" for w in ew]}), use_container_width=True)

    with st.expander("📋 Full Configuration Applied to This Run"):
        st.json({k: str(v) for k, v in IDEAL_CONFIGS[ticker].items()})

    with st.expander("📋 Trade Log (last 200 bars)"):
        st.dataframe(bt[['Predicted_Return','Uncertainty','Conviction','Position','Cost','Net_Return','Dir_Correct']].tail(200), use_container_width=True)
        st.download_button("⬇️ Download Backtest CSV", bt.to_csv().encode('utf-8'), f"backtest_{ticker}_{arch_short.replace(' ','_')}.csv", "text/csv")

Overwriting app.py


In [4]:
import subprocess

import time

import os

from google.colab import output

from google.colab.output import eval_js


print("Cleaning up old background processes...")

os.system("pkill -f streamlit")

time.sleep(1)


print("Starting Streamlit server...")

# 1. We removed the DEVNULL silencers so we can see if it crashes

# 2. We added back the 0.0.0.0 binding and CORS bypass

subprocess.Popen([

    "streamlit", "run", "app.py",

    "--server.headless", "true",

    "--server.address", "0.0.0.0",

    "--server.enableCORS", "false",

    "--server.enableXsrfProtection", "false"

])


# Give the server 5 full seconds to initialize before the proxy connects

time.sleep(5)


print("Opening native Colab proxy...")

proxy_url = eval_js("google.colab.kernel.proxyPort(8501)")

print(f"Click here for full-screen dashboard: {proxy_url}")


print("Loading embedded dashboard...")

output.serve_kernel_port_as_iframe(8501, height=1000)

Cleaning up old background processes...
Starting Streamlit server...
Opening native Colab proxy...
Click here for full-screen dashboard: https://8501-gpu-t4-s-kkb-usw1b2-m22dpk5uqheh-b.us-west1-2.prod.colab.dev
Loading embedded dashboard...


<IPython.core.display.Javascript object>